# Neural Network xG Model

In [1]:
import pandas as pd
from football_analytics.utils import parsing, supabase, shot_geometry
import football_analytics.visuals.shots as shots
from sklearn.metrics import log_loss
from importlib import reload
import numpy as np
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import joblib
reload(parsing)
reload(supabase)
reload(shot_geometry)
reload(shots)

<module 'football_analytics.visuals.shots' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\visuals\\shots.py'>

### Load Supabase Data

In [2]:
table_name = 'shots'
key_column = 'statsbomb_event_id'

In [3]:
data = supabase.fetch_all_rows_in_batches(table_name=table_name, key_column=key_column, batch_size=2000)

Fetched 2000 rows (total 2000).
Fetched 2000 rows (total 4000).
Fetched 2000 rows (total 6000).
Fetched 2000 rows (total 8000).
Fetched 2000 rows (total 10000).
Fetched 2000 rows (total 12000).
Fetched 2000 rows (total 14000).
Fetched 2000 rows (total 16000).
Fetched 2000 rows (total 18000).
Fetched 2000 rows (total 20000).
Fetched 2000 rows (total 22000).
Fetched 2000 rows (total 24000).
Fetched 2000 rows (total 26000).
Fetched 2000 rows (total 28000).
Fetched 2000 rows (total 30000).
Fetched 2000 rows (total 32000).
Fetched 2000 rows (total 34000).
Fetched 2000 rows (total 36000).
Fetched 2000 rows (total 38000).
Fetched 2000 rows (total 40000).
Fetched 2000 rows (total 42000).
Fetched 2000 rows (total 44000).
Fetched 2000 rows (total 46000).
Fetched 2000 rows (total 48000).
Fetched 2000 rows (total 50000).
Fetched 2000 rows (total 52000).
Fetched 2000 rows (total 54000).
Fetched 2000 rows (total 56000).
Fetched 2000 rows (total 58000).
Fetched 2000 rows (total 60000).
Fetched 2000 r

In [35]:
df_raw = pd.DataFrame(data)

## Prepare Data

In [37]:
df = df_raw.dropna().copy()

In [38]:
df['goal_outcome'] = df['outcome'].apply(lambda x: 1 if x=='Goal' else 0)
df['is_with_feet'] = df['body_part'].apply(lambda x: 1 if (x=='Right Foot' or x=='Left Foot') else 0)
df['is_penalty'] = df['shot_type'].apply(lambda x: 1 if x=='Penalty' else 0)

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 88015 entries, 0 to 88022
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   created_at                  88015 non-null  object 
 1   statsbomb_event_id          88015 non-null  object 
 2   x1                          88015 non-null  float64
 3   y1                          88015 non-null  float64
 4   distance_to_goal            88015 non-null  float64
 5   angle_to_goal_deg           88015 non-null  float64
 6   keeper_distance             88015 non-null  float64
 7   min_defender_distance       88015 non-null  float64
 8   avg_defender_distance       88015 non-null  float64
 9   num_def_in_shot_triangle    88015 non-null  int64  
 10  num_teammates_in_box        88015 non-null  int64  
 11  shot_to_min_def_ratio       88015 non-null  float64
 12  shot_type                   88015 non-null  object 
 13  body_part                   88015 no

In [40]:
y = df['goal_outcome'].values.astype(int)
X = np.column_stack([
    df['distance_to_goal'].values, 
    df['angle_to_goal_deg'].values, 
    df['keeper_distance'].values, 
    df['min_defender_distance'].values,
    df['num_teammates_in_box'].values,
    df['is_penalty'].values,
    df['num_def_in_shot_triangle'].values,
    df['frac_goal_uncovered'].values,
    df['keeper_is_in_shot_triangle'].values,
    df['is_with_feet'].values,
    df['under_pressure'].values,
                     ])

#### Check for NaN and Inf in Input / Output

In [41]:
if np.isnan(y).sum() > 0:
    raise ValueError("NaN values found in target variable y")
if np.isnan(X).sum() > 0:
    raise ValueError("NaN values found in feature matrix X")

## Define Loader, Model and Optimizer

In [42]:
scaler = StandardScaler().fit(X)
X = scaler.transform(X)

In [43]:
class ShotDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [44]:
dataset = ShotDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [45]:
import torch.nn as nn

class XGModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)  # output = logit
        )

    def forward(self, x):
        return self.net(x)


In [46]:
model = XGModel(input_dim=X.shape[1])

In [47]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [48]:
np.isnan(X).sum(), np.isinf(X).sum()

(np.int64(0), np.int64(0))

In [49]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss / len(loader):.4f}")


Epoch 01 | Loss: 0.3352
Epoch 02 | Loss: 0.2701
Epoch 03 | Loss: 0.2689
Epoch 04 | Loss: 0.2681
Epoch 05 | Loss: 0.2674
Epoch 06 | Loss: 0.2675
Epoch 07 | Loss: 0.2671
Epoch 08 | Loss: 0.2667
Epoch 09 | Loss: 0.2667
Epoch 10 | Loss: 0.2665
Epoch 11 | Loss: 0.2663
Epoch 12 | Loss: 0.2663
Epoch 13 | Loss: 0.2659
Epoch 14 | Loss: 0.2657
Epoch 15 | Loss: 0.2655
Epoch 16 | Loss: 0.2656
Epoch 17 | Loss: 0.2654
Epoch 18 | Loss: 0.2650
Epoch 19 | Loss: 0.2652
Epoch 20 | Loss: 0.2650
Epoch 21 | Loss: 0.2650
Epoch 22 | Loss: 0.2648
Epoch 23 | Loss: 0.2650
Epoch 24 | Loss: 0.2646
Epoch 25 | Loss: 0.2646
Epoch 26 | Loss: 0.2644
Epoch 27 | Loss: 0.2643
Epoch 28 | Loss: 0.2642
Epoch 29 | Loss: 0.2644
Epoch 30 | Loss: 0.2641


In [50]:
def xg_log_loss(y_true, y_pred):
    """
    Computes binary cross-entropy (log loss) for xG predictions.
    """
    return log_loss(y_true, y_pred)

In [51]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X, dtype=torch.float32))
    xg_probs = torch.sigmoid(logits).numpy().flatten()


In [52]:
xg_log_loss_nn = xg_log_loss(y, xg_probs)
print(f"NN xG log loss: {xg_log_loss_nn:.4f}")

NN xG log loss: 0.2646


## Save Model

In [53]:
date_str = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f"nn_xg_model_{date_str}.pth"
torch.save(model.state_dict(), f'../nn_models/{model_name}')
joblib.dump(scaler, f'../nn_models/scaler_{date_str}.save')

['../nn_models/scaler_20251222_175749.save']

In [54]:
df['nn_xg'] = xg_probs

In [55]:
df[['nn_xg', 'statsbomb_xg','statsbomb_event_id']].to_csv('nn_xg_model_predictions.csv', index=False)

## Check Differences

In [56]:
event_id = '0f0070e2-3143-4ae0-8065-a77fd54c2084'
shot = df[df['statsbomb_event_id'] == event_id]

In [57]:
shot['outcome']

5172    Post
Name: outcome, dtype: object

In [58]:
X_tmp = np.column_stack([
    shot['distance_to_goal'].values, 
    shot['angle_to_goal_deg'].values, 
    shot['keeper_distance'].values, 
    shot['min_defender_distance'].values,
    shot['num_teammates_in_box'].values,
    shot['is_penalty'].values,
    shot['num_def_in_shot_triangle'].values,
    shot['frac_goal_uncovered'].values,
    shot['keeper_is_in_shot_triangle'].values,
    shot['is_with_feet'].values,
    shot['under_pressure'].values,
                     ])

In [59]:
X_tmp

array([[3.92001276e+01, 3.01427806e-02, 3.82632983e+01, 1.72122050e+01,
        2.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.00000000e+00,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00]])

In [60]:
X_tmp = scaler.transform(X_tmp)

In [61]:
X_tmp

array([[ 2.28850013, -1.60740008,  2.47905674,  5.8003927 ,  0.93828034,
        -0.12532415, -0.7790535 ,  2.1590063 , -4.76962529,  0.4389715 ,
        -0.55787988]])

In [62]:
with torch.no_grad():
    logits = model(torch.tensor(X_tmp, dtype=torch.float32))
    xg_tmp = torch.sigmoid(logits).numpy().flatten()

In [63]:
xg_tmp

array([0.12229137], dtype=float32)

In [64]:
shot_dict = parsing.parse_json_field(shot.iloc[0]['full_json'])

In [67]:
shot_dict

{'id': '0f0070e2-3143-4ae0-8065-a77fd54c2084',
 'index': 1461,
 'period': 1,
 'timestamp': '00:38:43.510',
 'minute': 38,
 'second': 43,
 'type': {'id': 16, 'name': 'Shot'},
 'possession': 59,
 'possession_team': {'id': 905, 'name': 'Romania'},
 'play_pattern': {'id': 2, 'name': 'From Corner'},
 'team': {'id': 905, 'name': 'Romania'},
 'player': {'id': 10916, 'name': 'Nicolae Claudiu Stanciu'},
 'position': {'id': 15, 'name': 'Left Center Midfield'},
 'location': [120.1, 0.8],
 'duration': 1.761714,
 'related_events': ['41050306-f3a9-40c7-bbf9-c5835eabe490'],
 'shot': {'one_on_one': True,
  'statsbomb_xg': 0.00018,
  'end_location': [120.0, 38.6, 3.0],
  'body_part': {'id': 40, 'name': 'Right Foot'},
  'type': {'id': 61, 'name': 'Corner'},
  'outcome': {'id': 99, 'name': 'Post'},
  'technique': {'id': 93, 'name': 'Normal'},
  'freeze_frame': [{'location': [116.0, 43.6],
    'player': {'id': 16509, 'name': 'Viktor Tsygankov'},
    'position': {'id': 17, 'name': 'Right Wing'},
    'teamm

In [68]:
under_pressure = shot_dict.get('under_pressure', False)


In [69]:
under_pressure

False

In [31]:
bool(under_pressure)

False

In [ ]:
shots.plot_shot_details(shot_dict)